In [1]:
!pip install -q spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 87.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:



# =========================================================
# IMPORTS
# =========================================================

import json
import re
import spacy
import pandas as pd


# =========================================================
# LOAD SPACY MODEL
# =========================================================

nlp = spacy.load("en_core_web_sm")


# =========================================================
# DICTIONARIES
# =========================================================

FORM_DICT = {
    "tab": "tablet",
    "tablet": "tablet",
    "cap": "capsule",
    "capsule": "capsule",
    "syp": "syrup",
    "syrup": "syrup",
    "inj": "injection"
}

FREQ_LIST = [
    "OD",
    "BD",
    "TID",
    "TDS",
    "QID",
    "HS",
    "SOS"
]

PURPOSE_WORDS = [
    "allergy",
    "pain",
    "fever",
    "diarrhoea",
    "cough",
    "cold",
    "infection",
    "vomiting",
    "acidity"
]

NOTE_WORDS = [
    "after food",
    "before food",
    "at bedtime",
    "morning only",
    "night only",
    "empty stomach"
]


# =========================================================
# REGEX PATTERNS
# =========================================================

STRENGTH_RE = r'\b\d+(\.\d+)?\s?(mg|ml|mcg|g|iu)\b'

DOSAGE_RE = r'\b\d+\s?(tablet|tab|capsule|cap|ml|drop|drops)\b'


# =========================================================
# FORM EXTRACTION
# =========================================================

def extract_form(text):

    for short, full in FORM_DICT.items():

        pattern = r'\b' + re.escape(short) + r'\.?\b'

        if re.search(pattern, text, re.IGNORECASE):

            return full

    return ""


# =========================================================
# STRENGTH EXTRACTION
# =========================================================

def extract_strength(text):

    match = re.search(STRENGTH_RE, text, re.IGNORECASE)

    if match:
        return match.group(0)

    return ""


# =========================================================
# DOSAGE EXTRACTION
# =========================================================

def extract_dosage(text):

    match = re.search(DOSAGE_RE, text, re.IGNORECASE)

    if match:
        return match.group(0)

    return ""


# =========================================================
# FREQUENCY EXTRACTION
# =========================================================

def extract_frequency(text):

    for freq in FREQ_LIST:

        if re.search(r'\b' + freq + r'\b',
                     text,
                     re.IGNORECASE):

            return freq

    return ""


# =========================================================
# DURATION NORMALIZATION
# =========================================================

def normalize_duration_unit(num, unit):

    unit = unit.lower()

    if unit in ["d", "day", "days"]:

        return f"{num} days"

    if unit in ["w", "week", "weeks"]:

        return f"{num} weeks"

    if unit in ["m", "month", "months"]:

        return f"{num} months"

    return ""


# =========================================================
# DURATION EXTRACTION
# =========================================================

def extract_duration(text):

    text = text.lower()

    patterns = [

        r'for\s+(\d+)\s*(day|days|week|weeks|month|months)',

        r'\b(\d+)\s*(day|days|week|weeks|month|months)\b',

        r'\bx\s*(\d+)\s*([dwm])\b',

        r'\b(\d+)\s*([dwm])\b'
    ]

    for pattern in patterns:

        match = re.search(pattern, text)

        if match:

            num = match.group(1)

            unit = match.group(2)

            return normalize_duration_unit(num, unit)

    return ""


# =========================================================
# PURPOSE EXTRACTION
# =========================================================

def extract_purpose(text):

    purposes = []

    for purpose in PURPOSE_WORDS:

        if re.search(r'\b' + re.escape(purpose) + r'\b',
                     text,
                     re.IGNORECASE):

            purposes.append(purpose)

    return ", ".join(purposes)


# =========================================================
# NOTES EXTRACTION
# =========================================================

def extract_notes(text):

    notes = []

    for note in NOTE_WORDS:

        if re.search(re.escape(note),
                     text,
                     re.IGNORECASE):

            notes.append(note)

    return ", ".join(notes)


# =========================================================
# MEDICINE NAME EXTRACTION
# =========================================================

def extract_medicine_name(text):

    doc = nlp(text)

    medicine_tokens = []

    stop_words = set()

    # forms
    stop_words.update(FORM_DICT.keys())

    # frequencies
    stop_words.update([x.lower() for x in FREQ_LIST])

    # purposes
    stop_words.update(PURPOSE_WORDS)

    # notes
    for note in NOTE_WORDS:

        for w in note.split():

            stop_words.add(w.lower())

    # common instruction words
    stop_words.update([
        "for",
        "days",
        "day",
        "weeks",
        "week",
        "months",
        "month",
        "only"
    ])

    for token in doc:

        token_text = token.text
        lower = token_text.lower()

        # stop once dosage/strength starts
        if re.fullmatch(r'\d+(\.\d+)?', token_text):

            break

        # skip punctuation
        if token.is_punct:
            continue

        # skip stop words
        if lower in stop_words:
            continue

        # skip units
        if lower in ["mg", "ml", "mcg", "g", "iu"]:
            continue

        # valid medicine token
        if re.match(r'^[A-Za-z0-9\-]+$', token_text):

            medicine_tokens.append(token_text)

    return " ".join(medicine_tokens).strip()


# =========================================================
# MAIN PARSER
# =========================================================

def parse_prescription(raw_text):

    text = raw_text.strip()

    result = {

        "raw_text": raw_text,

        "medicine_name": extract_medicine_name(text),

        "form": extract_form(text),

        "strength": extract_strength(text),

        "dosage": extract_dosage(text),

        "frequency": extract_frequency(text),

        "duration": extract_duration(text),

        "purpose": extract_purpose(text),

        "notes": extract_notes(text)
    }

    return result


# =========================================================
# FILE PATHS
# =========================================================

INPUT_FILE = "/content/prescription_raw_text_only.json"

OUTPUT_FILE = "structured_output.json"


# =========================================================
# LOAD INPUT
# =========================================================

with open(INPUT_FILE, "r", encoding="utf-8") as f:

    data = json.load(f)


# =========================================================
# PROCESS RECORDS
# =========================================================

results = []

for row in data:

    raw_text = row["raw_text"]

    parsed = parse_prescription(raw_text)

    results.append(parsed)


# =========================================================
# SAVE OUTPUT
# =========================================================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    json.dump(results, f, indent=4)


# =========================================================
# PRINT OUTPUT
# =========================================================

print("\nProcessing Completed.")

print(f"\nSaved structured output to: {OUTPUT_FILE}")

print("\nSample Structured Output:\n")

print(json.dumps(results[:5], indent=4))


# =========================================================
# SIMPLE EXTRACTION ACCURACY
# =========================================================
#
# This is heuristic accuracy:
# percentage of records where key fields
# were extracted successfully
#
# =========================================================

total = len(results)

medicine_ok = 0
strength_ok = 0
frequency_ok = 0
duration_ok = 0

for r in results:

    if r["medicine_name"]:
        medicine_ok += 1

    if r["strength"]:
        strength_ok += 1

    if r["frequency"]:
        frequency_ok += 1

    if r["duration"]:
        duration_ok += 1


print("\n================ Accuracy Summary ================\n")

print(f"Total Records: {total}")

print(f"Medicine Extraction Accuracy : {(medicine_ok/total)*100:.2f}%")

print(f"Strength Extraction Accuracy : {(strength_ok/total)*100:.2f}%")

print(f"Frequency Extraction Accuracy: {(frequency_ok/total)*100:.2f}%")

print(f"Duration Extraction Accuracy : {(duration_ok/total)*100:.2f}%")

overall = (
    medicine_ok +
    strength_ok +
    frequency_ok +
    duration_ok
) / (total * 4)

print(f"\nOverall Heuristic Accuracy: {overall*100:.2f}%")


Processing Completed.

Saved structured output to: structured_output.json

Sample Structured Output:

[
    {
        "raw_text": "Fexofenadin 120 mg OD allergy 7 days",
        "medicine_name": "Fexofenadin",
        "form": "",
        "strength": "120 mg",
        "dosage": "",
        "frequency": "OD",
        "duration": "7 days",
        "purpose": "allergy",
        "notes": ""
    },
    {
        "raw_text": "Tab. Amitriptyline 25 mg 1 tablet OD for 5 days at bedtime",
        "medicine_name": "Amitriptyline",
        "form": "tablet",
        "strength": "25 mg",
        "dosage": "1 tablet",
        "frequency": "OD",
        "duration": "5 days",
        "purpose": "",
        "notes": "at bedtime"
    },
    {
        "raw_text": "Tab Ranitidine 150 mg 1 tablet BD 10 Days at bedtime",
        "medicine_name": "Ranitidine",
        "form": "tablet",
        "strength": "150 mg",
        "dosage": "1 tablet",
        "frequency": "BD",
        "duration": "10 days",
      